# Análise do Pipeline KEV

Notebook de leitura e validação dos artefatos gerados pelo pacote `kev_pipeline`.

Uso recomendado:

- deixe `EXECUTE_PIPELINE=False` para modo offline
- use `EXECUTE_PIPELINE=True` apenas quando quiser rodar o pipeline a partir do notebook
- valide métricas, deltas e enriquecimentos nas células seguintes


In [ ]:
# 1) configuração do notebook e do pipeline
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path.cwd()
SRC = ROOT / "src"
ARTIFACTS_DIR = Path("artifacts")
MPL_CACHE_DIR = Path(".cache_mpl")
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from kev_pipeline import PipelineConfig, run_pipeline
from kev_pipeline.env import load_dotenv

load_dotenv()

EXECUTE_PIPELINE = False
PIPELINE_MODE = "kev"
RUN_NVD = False
RUN_EPSS = False
RUN_GITHUB_ADVISORIES = False
SNAPSHOT_DATE = None  # ex.: "2026-03-29"
GENERATE_PLOTS = True
NVD_API_KEY = os.getenv("NVD_API_KEY", "")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")

OUT_DIR = ARTIFACTS_DIR / "current"
SNAPSHOTS_DIR = ARTIFACTS_DIR / "snapshots"
DELTAS_DIR = ARTIFACTS_DIR / "deltas"

print(f"Executar pipeline: {EXECUTE_PIPELINE}")
print(f"Pipeline mode: {PIPELINE_MODE}")
print(f"NVD ativo: {RUN_NVD}")
print(f"EPSS ativo: {RUN_EPSS}")
print(f"GitHub Advisories ativo: {RUN_GITHUB_ADVISORIES}")
print(f"Diretório de artefatos: {ARTIFACTS_DIR}")


In [ ]:
# 2) helpers de leitura dos artefatos gerados pelo pacote

def load_csv_if_exists(path):
    if not path:
        return pd.DataFrame()

    path = Path(path)
    if not path.exists() or not path.is_file():
        return pd.DataFrame()

    return pd.read_csv(path)


def load_json_if_exists(path):
    if not path:
        return {}

    path = Path(path)
    if not path.exists() or not path.is_file():
        return {}

    return json.loads(path.read_text(encoding="utf-8"))


def first_existing_path(*paths):
    for path in paths:
        if not path:
            continue
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def summary_view(summary):
    return {
        "snapshot_date": summary.get("snapshot_date", ""),
        "pipeline_mode": summary.get("pipeline_mode", ""),
        "rows": summary.get("rows", {}),
        "delta_counts": summary.get("delta", {}).get("counts", {}),
        "warnings": summary.get("warnings", []),
        "enrichment_failures": summary.get("enrichment_failures", {}).get("count", 0),
        "summary": summary.get("files", {}).get("summary", ""),
        "snapshot_dir": summary.get("files", {}).get("snapshot_dir", ""),
        "delta_dir": summary.get("files", {}).get("delta_dir", ""),
    }


def build_config():
    snapshot_date = None
    if SNAPSHOT_DATE:
        snapshot_date = pd.to_datetime(SNAPSHOT_DATE).date()

    return PipelineConfig(
        pipeline_mode=PIPELINE_MODE,
        run_nvd=RUN_NVD,
        run_epss=RUN_EPSS,
        run_github_advisories=RUN_GITHUB_ADVISORIES,
        out_dir=OUT_DIR,
        snapshots_dir=SNAPSHOTS_DIR,
        deltas_dir=DELTAS_DIR,
        nvd_api_key=NVD_API_KEY,
        github_token=GITHUB_TOKEN,
        generate_plots=GENERATE_PLOTS,
        snapshot_date=snapshot_date or PipelineConfig().snapshot_date,
    )


In [ ]:
# 3) execução principal via pacote kev_pipeline
if EXECUTE_PIPELINE:
    config = build_config()
    summary = run_pipeline(config)
    summary_path = Path(summary["files"]["summary"])
else:
    summary_path = OUT_DIR / "summary.json"
    summary = load_json_if_exists(summary_path)
    if not summary:
        snapshot_summaries = sorted((ARTIFACTS_DIR / "snapshots").glob("*/summary.json"))
        if snapshot_summaries:
            summary_path = snapshot_summaries[-1]
            summary = load_json_if_exists(summary_path)
    if not summary:
        raise RuntimeError(
            "Nenhum summary.json encontrado em artifacts/current nem em artifacts/snapshots. "
            "Defina EXECUTE_PIPELINE=True para gerar os artefatos ou execute o pipeline antes."
        )

snapshot_dir = Path(summary["files"]["snapshot_dir"])
delta_dir = Path(summary["files"]["delta_dir"])

threats_daily_events_df = load_csv_if_exists(first_existing_path(summary["files"].get("threats_daily_events"), snapshot_dir / "threats_daily_events.csv"))
threats_daily_counts_df = load_csv_if_exists(first_existing_path(summary["files"].get("threats_daily_counts"), snapshot_dir / "threats_daily_counts.csv"))
threats_by_vendor_df = load_csv_if_exists(first_existing_path(summary["files"].get("threats_by_vendor"), snapshot_dir / "threats_by_vendor.csv"))
threats_by_product_df = load_csv_if_exists(first_existing_path(summary["files"].get("threats_by_product"), snapshot_dir / "threats_by_product.csv"))
enrich_nvd_df = load_csv_if_exists(first_existing_path(summary["files"].get("enrich_nvd"), snapshot_dir / "enrich_nvd.csv"))
enrich_epss_df = load_csv_if_exists(first_existing_path(summary["files"].get("enrich_epss"), snapshot_dir / "enrich_epss.csv"))
enrich_github_advisories_df = load_csv_if_exists(first_existing_path(summary["files"].get("enrich_github_advisories"), snapshot_dir / "enrich_github_advisories.csv"))
threats_daily_enriched_df = load_csv_if_exists(first_existing_path(summary["files"].get("threats_daily_enriched"), snapshot_dir / "threats_daily_enriched.csv"))
new_cves_today_df = load_csv_if_exists(delta_dir / "new_cves_today.csv")
new_urgent_today_df = load_csv_if_exists(delta_dir / "new_urgent_today.csv")
new_ransomware_today_df = load_csv_if_exists(delta_dir / "new_ransomware_today.csv")

print("Resumo da execução")
print(json.dumps(summary_view(summary), indent=2, ensure_ascii=False))


## Análise e validação

Revise volume total, intervalo temporal, itens urgentes, `ransomware_flag`, agregações e delta em relação ao snapshot anterior.


In [ ]:
# 4) análise e validação
if threats_daily_events_df.empty:
    raise RuntimeError("Execute a célula 3 (execução principal) antes desta análise.")

metrics = {
    "rows_threats_daily_events": int(len(threats_daily_events_df)),
    "rows_threats_daily_counts": int(len(threats_daily_counts_df)),
    "unique_days": int(threats_daily_events_df["date"].nunique()) if "date" in threats_daily_events_df.columns else 0,
    "ransomware_flag_sum": int(threats_daily_events_df["ransomware_flag"].sum()) if "ransomware_flag" in threats_daily_events_df.columns else 0,
    "urgent_sum": int(threats_daily_events_df["urgent"].sum()) if "urgent" in threats_daily_events_df.columns else 0,
    "min_date": "" if threats_daily_events_df.empty else str(threats_daily_events_df["date"].min()),
    "max_date": "" if threats_daily_events_df.empty else str(threats_daily_events_df["date"].max()),
    "delta_new_cves_today": int(len(new_cves_today_df)),
    "delta_new_urgent_today": int(len(new_urgent_today_df)),
    "delta_new_ransomware_today": int(len(new_ransomware_today_df)),
}

print("Métricas principais")
print(json.dumps(metrics, indent=2, ensure_ascii=False))

print()
print("Top vendors")
display(threats_by_vendor_df.head(15))

print()
print("Top products")
display(threats_by_product_df.head(15))

print()
print("Novos CVEs do delta")
display(new_cves_today_df.head(10))


In [ ]:
# 5) visualizações rápidas a partir dos CSVs gerados
if threats_daily_counts_df.empty:
    raise RuntimeError("Execute a célula 3 (execução principal) antes dos gráficos.")

import matplotlib.pyplot as plt
import plotly.express as px

fig_daily = px.line(
    threats_daily_counts_df,
    x="date",
    y="threat_count",
    title="Ameaças adicionadas por dia (KEV)",
)
display(HTML(fig_daily.to_html(include_plotlyjs="cdn", full_html=False)))

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(threats_daily_counts_df["date"]), threats_daily_counts_df["threat_count"], linewidth=1.2)
plt.title("Ameaças adicionadas por dia (KEV)")
plt.xlabel("Data")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()


In [ ]:
# 6) resumo final e localização dos artefatos
print("Resumo final")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print()
print("Snapshot atual:")
print(snapshot_dir)

print()
print("Delta atual:")
print(delta_dir)


# 6) resumo final e localização dos artefatos
print("Resumo final")
print(json.dumps(summary_view(summary), indent=2, ensure_ascii=False))

print()
print("Snapshot atual:")
print(summary.get("files", {}).get("snapshot_dir", ""))

print()
print("Delta atual:")
print(summary.get("files", {}).get("delta_dir", ""))


In [ ]:
# 7) validação da seção opcional de enriquecimento
print("Warnings:")
print(json.dumps(summary.get("warnings", []), indent=2, ensure_ascii=False))

print()
print("Falhas de enriquecimento:")
print(json.dumps(summary.get("enrichment_failures", {}), indent=2, ensure_ascii=False))

if not enrich_nvd_df.empty:
    print()
    print(f"NVD enrichment disponível: {len(enrich_nvd_df):,} linhas")
    display(enrich_nvd_df.head(10))
else:
    print()
    print("NVD sem dados nesta execução (ou desativado).")

if not enrich_epss_df.empty:
    print()
    print(f"EPSS enrichment disponível: {len(enrich_epss_df):,} linhas")
    display(enrich_epss_df.head(10))
else:
    print()
    print("EPSS sem dados nesta execução (ou desativado).")

if not enrich_github_advisories_df.empty:
    print()
    print(f"GitHub Advisories enrichment disponível: {len(enrich_github_advisories_df):,} linhas")
    display(enrich_github_advisories_df.head(10))
else:
    print()
    print("GitHub Advisories sem dados nesta execução (ou desativado).")

if not threats_daily_enriched_df.empty:
    print()
    print(f"Base enriquecida disponível: {len(threats_daily_enriched_df):,} linhas")
    display(threats_daily_enriched_df.head(10))
else:
    print()
    print("Base enriquecida não foi gerada nesta execução.")
